This notebook is an test from the TabDDPM paper. I will adapt the code from TabDDPM to match the Katebatic system later.

In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import tomli
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import zero
import lib

# Import pipeline functions
from scripts.train import train
from scripts.sample import sample
from scripts.eval_catboost import train_catboost
from scripts.eval_mlp import train_mlp
from scripts.eval_simple import train_simple

# Configuration Loading

In [2]:
def load_config(config_path):
    """Load TOML configuration file"""
    with open(config_path, 'rb') as f:
        return tomli.load(f)

def save_file(parent_dir, config_path):
    """Save configuration file to output directory"""
    try:
        dst = os.path.join(parent_dir, 'config.toml')
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copyfile(os.path.abspath(config_path), dst)
    except shutil.SameFileError:
        pass

In [3]:
# Load your configuration
config_path = "exp/insurance/ddpm_cb_best/config.toml"
raw_config = load_config(config_path)

# Set device
if 'device' in raw_config:
    device = torch.device(raw_config['device'])
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")
print(f"Dataset: {raw_config.get('real_data_path', 'Unknown')}")

Using device: cuda:0
Dataset: data/insurance/


# Training
This is where we train the TabDDPM dataset for it to understand our input data. After that, we generate synthetic dataset.

What happens:
- Load the real dataset `data/insurance/`
- Train TabDDPM to learn the real dataset
- Save the model to `exp/insurance/ddpm_cb_best/model.pt`

In [4]:
def tabddpm_training(config, device, change_val=False):
    """Run TabDDPM training"""
    timer = zero.Timer()
    timer.run()
    
    # Save config file
    save_file(config['parent_dir'], config_path)
    
    print("Starting training...")
    train(
        **config['train']['main'],
        **config['diffusion_params'],
        parent_dir=config['parent_dir'],
        real_data_path=config['real_data_path'],
        model_type=config['model_type'],
        model_params=config['model_params'],
        T_dict=config['train']['T'],
        num_numerical_features=config['num_numerical_features'],
        device=device,
        change_val=change_val
    )
    
    print(f'Training completed in: {str(timer)}')
    return timer

In [4]:
def fix_none_values(config):
    """Convert '__none__' strings to actual None values"""
    def convert_none(obj):
        if isinstance(obj, dict):
            return {k: convert_none(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_none(item) for item in obj]
        elif obj == "__none__":
            return None
        else:
            return obj
    
    return convert_none(config)

# Apply the fix to your config
raw_config = fix_none_values(raw_config)

# Verify the fix worked
print("Fixed train.T section:")
for key, value in raw_config['train']['T'].items():
    print(f"  {key}: {value} ({type(value).__name__})")

print("\nFixed eval.T section:")
for key, value in raw_config['eval']['T'].items():
    print(f"  {key}: {value} ({type(value).__name__})")

Fixed train.T section:
  seed: 0 (int)
  normalization: quantile (str)
  num_nan_policy: None (NoneType)
  cat_nan_policy: None (NoneType)
  cat_min_frequency: None (NoneType)
  cat_encoding: None (NoneType)
  y_policy: default (str)

Fixed eval.T section:
  seed: 0 (int)
  normalization: None (NoneType)
  num_nan_policy: None (NoneType)
  cat_nan_policy: None (NoneType)
  cat_min_frequency: None (NoneType)
  cat_encoding: None (NoneType)
  y_policy: default (str)


In [7]:
# Now run training
training_timer = tabddpm_training(raw_config, device)

Starting training...
[2 2 4]
12
{'d_in': 12, 'num_classes': 0, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 512, 512, 512, 512, 256], 'dropout': 0.0}}
mlp
Step 500/30000 MLoss: 0.7906 GLoss: 0.855 Sum: 1.6456
Step 1000/30000 MLoss: 0.7429 GLoss: 0.5863 Sum: 1.3292000000000002
Step 1500/30000 MLoss: 0.7385 GLoss: 0.5769 Sum: 1.3154
Step 2000/30000 MLoss: 0.7362 GLoss: 0.4998 Sum: 1.236
Step 2500/30000 MLoss: 0.7334 GLoss: 0.4316 Sum: 1.165
Step 3000/30000 MLoss: 0.7173 GLoss: 0.414 Sum: 1.1313
Step 3500/30000 MLoss: 0.7109 GLoss: 0.4076 Sum: 1.1185
Step 4000/30000 MLoss: 0.7162 GLoss: 0.4059 Sum: 1.1220999999999999
Step 4500/30000 MLoss: 0.7094 GLoss: 0.4073 Sum: 1.1167
Step 5000/30000 MLoss: 0.6914 GLoss: 0.3972 Sum: 1.0886
Step 5500/30000 MLoss: 0.6968 GLoss: 0.3991 Sum: 1.0958999999999999
Step 6000/30000 MLoss: 0.6978 GLoss: 0.3942 Sum: 1.092
Step 6500/30000 MLoss: 0.6936 GLoss: 0.3991 Sum: 1.0927
Step 7000/30000 MLoss: 0.688 GLoss: 0.3921 Sum: 1.0800999999999998
Step 7500/3

# Sampling
This is when we generate synthetic dataset

What happens:
- Loads the trained model from `exp/insurance/ddpm_cb_best/model.pt`
- Uses reverse diffusion to generate synthetic samples
- Saves synthetic data as `exp/insurance/ddpm_cb_best`

In [5]:
def run_sampling(config, device, num_samples=None, change_val=False):
    """Generate synthetic samples"""
    timer = zero.Timer()
    timer.run()
    
    samples = num_samples or config['sample']['num_samples']
    
    print(f"Generating {samples} synthetic samples...")
    sample(
        num_samples=samples,
        batch_size=config['sample']['batch_size'],
        disbalance=config['sample'].get('disbalance', None),
        **config['diffusion_params'],
        parent_dir=config['parent_dir'],
        real_data_path=config['real_data_path'],
        model_path=os.path.join(config['parent_dir'], 'model.pt'),
        model_type=config['model_type'],
        model_params=config['model_params'],
        T_dict=config['train']['T'],
        num_numerical_features=config['num_numerical_features'],
        device=device,
        seed=config['sample'].get('seed', 0),
        change_val=change_val
    )
    
    # Save info file
    info_src = os.path.join(config['real_data_path'], 'info.json')
    info_dst = os.path.join(config['parent_dir'], 'info.json')
    if os.path.exists(info_src):
        shutil.copyfile(info_src, info_dst)
    
    print(f'Sampling completed in: {str(timer)}')
    return timer

In [6]:
# Run sampling
sampling_timer = run_sampling(raw_config, device)

Generating 7200 synthetic samples...
mlp
Sample timestep    0
Discrete cols: [2]
Num shape:  (7200, 3)
Sampling completed in: 0:00:06


# Load and Inspect data
Training reads real data from `data/insurance/`:
- `X_num_train.npy`, `X_cat_train.npy`, `y_train.npy`
- Learns the distribution
- Saves model to `exp/insurance/ddpm_cb_best/model.pt`

Sampling creates synthetic data in `exp/insurance/ddpm_cb_best/`:
- Generates synthetic versions: `X_num_train.npy`, `X_cat_train.npy`, etc.

Evaluation compares:
- Train model on synthetic data from `exp/insurance/ddpm_cb_best/`
- Test on real test data from `data/insurance/`

In [7]:
def load_and_compare_data(config):
    """Load real and synthetic data for comparison"""
    
    real_path = config['real_data_path']
    synthetic_path = config['parent_dir']
    
    print("Loading data for comparison...")
    
    # Load real data
    if os.path.exists(os.path.join(real_path, 'X_num_train.npy')):
        real_X_num = np.load(os.path.join(real_path, 'X_num_train.npy'))
        real_y = np.load(os.path.join(real_path, 'y_train.npy'))
        print(f"✓ Real data: {real_X_num.shape[0]} samples, {real_X_num.shape[1]} numerical features")
    
    # Load synthetic data (after sampling)
    synthetic_files = []
    if os.path.exists(synthetic_path):
        if os.path.exists(os.path.join(synthetic_path, 'X_num_train.npy')):
            synthetic_X_num = np.load(os.path.join(synthetic_path, 'X_num_train.npy'))
            synthetic_y = np.load(os.path.join(synthetic_path, 'y_train.npy'))
            print(f"✓ Synthetic data: {synthetic_X_num.shape[0]} samples, {synthetic_X_num.shape[1]} numerical features")
            
            # Quick comparison
            print(f"\nQuick comparison:")
            print(f"  Real mean: {real_X_num.mean(axis=0)[:3]}...")
            print(f"  Synthetic mean: {synthetic_X_num.mean(axis=0)[:3]}...")
        else:
            print("✗ No synthetic data found yet (run sampling first)")

# Check your data
load_and_compare_data(raw_config)

Loading data for comparison...
✓ Real data: 856 samples, 3 numerical features
✓ Synthetic data: 7200 samples, 3 numerical features

Quick comparison:
  Real mean: [39.16121495 30.58464369  1.07827103]...
  Synthetic mean: [39.40844298 30.47686858  1.04347222]...


# Multi-Seed Evaluation
Instead of running evaluation once, multi-seed runs the same evaluation multiple times with different random seeds and reports the mean ± standard deviation of the results.

In [8]:
# Import the eval_seeds function
from scripts.eval_seeds import eval_seeds

def run_multi_seed_evaluation(config, n_seeds=5, eval_model='catboost', 
                             eval_type='synthetic', n_datasets=1, change_val=False):
    """Run robust multi-seed evaluation"""
    timer = zero.Timer()
    timer.run()
    
    print(f"Running {eval_model} evaluation with {n_seeds} seeds...")
    print(f"Evaluation type: {eval_type}")
    print(f"Number of datasets: {n_datasets}")
    
    # Run multi-seed evaluation
    results = eval_seeds(
        raw_config=config,
        n_seeds=n_seeds,
        eval_type=eval_type,
        sampling_method="ddpm",
        model_type=eval_model,
        n_datasets=n_datasets,
        dump=True,
        change_val=change_val
    )
    
    print(f'Multi-seed evaluation completed in: {str(timer)}')
    return results, timer

In [10]:
# Run CatBoost evaluation with 5 seeds
catboost_results, catboost_timer = run_multi_seed_evaluation(
    raw_config, 
    n_seeds=10, # Number of random seeds for evaluation
    eval_model='catboost', 
    eval_type='synthetic',
    n_datasets=5 # Number of different synthetic datasets
)

print("\nCatBoost Results:")
for metric, values in catboost_results.items():
    if isinstance(values, dict) and 'mean' in values:
        print(f"  {metric}: {values['mean']:.4f} ± {values['std']:.4f}")
    else:
        print(f"  {metric}: {values}")

Running catboost evaluation with 10 seeds...
Evaluation type: synthetic
Number of datasets: 5
Copying existing synthetic data to temp directory...
  ✓ Copied X_num_train.npy
  ✓ Copied X_cat_train.npy
  ✓ Copied y_train.npy
  - Skipped X_num_test.npy (not found)
  - Skipped X_cat_test.npy (not found)
  - Skipped y_test.npy (not found)
  - Skipped X_num_val.npy (not found)
  - Skipped X_cat_val.npy (not found)
  - Skipped y_val.npy (not found)
**Eval Iter: 1/50**
----------------------------------------------------------------------------------------------------
loading synthetic data: /tmp/tmpefbc8egt
Train size: (7200, 6), Val size (214, 6)
{'seed': 0, 'normalization': None, 'num_nan_policy': None, 'cat_nan_policy': None, 'cat_min_frequency': None, 'cat_encoding': None, 'y_policy': 'default'}
{'bagging_temperature': 0.9636627605010293,
 'cat_features': [3, 4, 5],
 'depth': 6,
 'early_stopping_rounds': 50,
 'iterations': 2000,
 'l2_leaf_reg': 8.92855270774259,
 'leaf_estimation_iterati